In [2]:
import re 
import os 
import numpy as np 
import pandas as pd
import torch 
import unicodedata #standardize/normalize the text 
from tokenizers import Tokenizer #Tokenizer class from the tokenizers library
from tokenizers.models import BPE #byte-pair encoding 
from tokenizers.trainers import BpeTrainer 
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

In [3]:
#Special tokens in the vocab 
SPECIAL_TOKENS = ["<pad>", "<unk>", "<sos>", "<eos>"]
PAD_IDX, UNK_IDX, SOS_IDX, EOS_IDX = 0, 1, 2, 3

In [4]:
#NFC is Normalization Form C.
def normalize_text(text):
    if not isinstance(text, str):
        text = str(text)
    # remove invisible characters 
    text = text.replace('\u200d', '').replace('\u200c', '')
    # replace one or more consecutive spaces into one single space 
    text = re.sub(r' +', ' ', text).strip() 
    # finally normalize for unique token ids for similar words 
    text = unicodedata.normalize('NFC', text)
    return text

In [5]:
'''
If the vocabulary size is small, 
there is a higher chance that a word won't exist as a single token, 
so BPE splits it into smaller subwords.

If the vocabulary size is large, 
there is a higher chance that the complete word already exists in the vocabulary, 
so fewer or no splits are needed.
'''
def train_bpe_tokenizer(texts, vocab_size, save_dir):
    # tokenizer object (tokenizes based on BPE)
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    # handle raw text by splitting with whitespace 
    tokenizer.pre_tokenizer = Whitespace()
    # tokenizer trainer 
    trainer = BpeTrainer(special_tokens=SPECIAL_TOKENS, vocab_size=vocab_size)
    #train our BPE tokenizer 'tokenizer' on vocab 'texts' 
    tokenizer.train_from_iterator(texts, trainer)
    # final tokenizer object 
    fast_tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer,
        pad_token="<pad>", unk_token="<unk>",
        bos_token="<sos>", eos_token="<eos>",
    )
    # save final tokenizer so can be used later w/o retraining entire tokenizer 
    fast_tokenizer.save_pretrained(save_dir)
    return fast_tokenizer

In [6]:
def build_glove_vocab(texts, glove_path, embed_dim=100, special_tokens=SPECIAL_TOKENS, max_vocab=30000, min_freq=2):
    # freq map of words in the texts dataset 
    word_freqs = {}
    for t in texts:
        tokens = str(t).split()
        for tok in tokens:
            word_freqs[tok] = word_freqs.get(tok, 0) + 1
    # consider only the words which appear atleast 'min_freq' times in the dataset
    valid_words = [w for w, freq in word_freqs.items() if freq >= min_freq]
    # sort based on most frequently occuring first 
    valid_words.sort(key=lambda w: word_freqs[w], reverse=True)
    # take top 'most frequently' occuring words to satisfy max_vocab size 
    valid_words = valid_words[:max_vocab - len(special_tokens)]
    # final words 
    corpus_words = set(valid_words)

    word2idx = {tok: i for i, tok in enumerate(special_tokens)}
    '''
    "<pad>": 0,
    "<unk>": 1,
    "<sos>": 2,
    "<eos>": 3
    '''
    vectors = [np.zeros(embed_dim, dtype=np.float32)] # <pad>
    for _ in range(1, len(special_tokens)):
        vectors.append(np.random.normal(scale=0.1, size=(embed_dim,)).astype(np.float32))
    '''
    embed_dim = 4 (say) 
    vectors = [
        [ 0.00,  0.00,  0.00,  0.00],   # <pad>
        [-0.08,  0.13,  0.02, -0.11],   # <unk>
        [ 0.05, -0.04,  0.19,  0.07],   # <sos>
        [-0.09,  0.01, -0.12,  0.15]    # <eos>
    ]
    '''
    # path to a pretrained GloVe word embedding file 
    # if word in corpus_word is already present in pretrained GloVe embedding matrix 
    # directly use the pretrained embedding for that word 
    # otherwise initialise a random embedding for that word 
    if glove_path and os.path.exists(glove_path):
        with open(glove_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.rstrip().split(" ")
                word = parts[0]
                if word in corpus_words and word not in word2idx:
                    word2idx[word] = len(word2idx)
                    vectors.append(np.array(parts[1:], dtype=np.float32))

    for word in corpus_words:
        if word not in word2idx:
            word2idx[word] = len(word2idx)
            vectors.append(np.random.normal(scale=0.1, size=(embed_dim,)).astype(np.float32))
    # final embedding matrix to get token's embeddings 
    embedding_matrix = torch.tensor(np.stack(vectors), dtype=torch.float32)
    return word2idx, embedding_matrix

In [7]:
if __name__ == "__main__":
    DATA_DIR = "./nmtdataset"
    HINDI_TRAIN = os.path.join(DATA_DIR, "Hindi_train.csv")
    MARATHI_TRAIN = os.path.join(DATA_DIR, "Marathi_train.csv")
    
    def clean_csv(path):
        if not os.path.exists(path): return []
        df = pd.read_csv(path)
        df["source"] = df["source"].astype(str).str.replace('translate to Hindi: ', '').str.replace('translate to Marathi: ', '')
        df["source"] = df["source"].apply(normalize_text)
        df["translated"] = df["translated"].astype(str).apply(normalize_text)
        return df

    hi_df = clean_csv(HINDI_TRAIN)
    mr_df = clean_csv(MARATHI_TRAIN)

    hi_texts = hi_df["translated"].tolist() if len(hi_df) else []
    mr_texts = mr_df["translated"].tolist() if len(mr_df) else []
    
    combined_target_texts = hi_texts + mr_texts
    if combined_target_texts:
        print("Training Combined Target BPE Tokenizer...")
        train_bpe_tokenizer(combined_target_texts, vocab_size=8000, save_dir="tokenizer_target")
        print("Combined Target Tokenizer saved to tokenizer_target/")

    hi_src = hi_df["source"].tolist() if len(hi_df) else []
    mr_src = mr_df["source"].tolist() if len(mr_df) else []
    combined_src_texts = hi_src + mr_src

    GLOVE_PATH = os.path.join(DATA_DIR, "glove.6B.100d.txt")
    if combined_src_texts:
        print("\nBuilding GloVe vocab from source English sentences...")
        word2idx, embedding_matrix = build_glove_vocab(
            combined_src_texts, GLOVE_PATH, embed_dim=100, 
            max_vocab=30000, min_freq=2
        )
        torch.save({"word2idx": word2idx, "matrix": embedding_matrix}, "glove_vocab.pt")
        print(f"GloVe vocab built: {len(word2idx)} words. Saved to glove_vocab.pt")

Training Combined Target BPE Tokenizer...
Combined Target Tokenizer saved to tokenizer_target/

Building GloVe vocab from source English sentences...
GloVe vocab built: 29157 words. Saved to glove_vocab.pt


In [1]:
'''Note that - 
BPE training repeatedly scans all Hindi and Marathi sentences, finds the most frequent neighboring symbol pair, merges it into a new token, and continues until the vocabulary contains 8000 learned tokens.
'''

'Note that - \nBPE training repeatedly scans all Hindi and Marathi sentences, finds the most frequent neighboring symbol pair, merges it into a new token, and continues until the vocabulary contains 8000 learned tokens.\n'